<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/Homework/HW08-text_as_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 8: A Spam Filter That Reads

**DSC 210 Foundations of Data Science**

*Covers Module 12 (text as data).*

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

References:
- Module 12 course notebook (cleaning and tokens, stop words, TF-IDF, a text classifier, n-grams, search, clustering, next-word models)
- T. A. Almeida, J. M. Gómez Hidalgo, and A. Yamakami (2011), "Contributions to the Study of SMS Spam Filtering: New Collection and Results," *Proceedings of the 11th ACM Symposium on Document Engineering (DocEng '11)*. The SMS Spam Collection is distributed through the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).
- `csv_data/sms_spam.csv`: the SMS Spam Collection with 309 messages removed that contain profanity, sexual content, or drug references (see `csv_data/sms_spam_README.txt`).

*Last major revision: 2026-11-28*

## Key Concepts

- Clean and tokenize raw text, and decide what cleaning should keep
- Decide, from the data and the task, whether to remove stop words
- Compute TF-IDF by hand, and build and read a TF-IDF + logistic regression classifier
- Search a collection with cosine similarity, and group it with k-means
- Build a next-word model, and explain when it starts copying its training text

## Instructions

**Total: 50 points.** Point values are listed with each problem.

- Cells marked `# RUN-TOGETHER` are complete; run them and read the output.
- Cells marked `# FILL-IN` contain blanks written as `____`.
- Cells marked `# ADAPT` are for you to write. Each one names a cell in the **Module 12 notebook**. Find that cell, copy it here, and change it to fit this problem. Use the variable names the problem asks for, because later cells use them.
- Run the cells **in order**.
- Most of the credit rides on your written answers. Quote specific numbers from your output.
- Submit the completed `.ipynb`, with all cells run, to Canvas.

---
## Part 0. Version 5 of the spam filter (2 points)
---

In Homework 6 you built a spam filter from six numbers per email: how many `!` and `$` signs, capital letters, and a few chosen words. The words themselves were gone. This time the filter gets to **read**.

The data is 5,263 real text messages (SMS), each labeled `spam` or `ham` (an ordinary message). The text is exactly as the senders typed it: abbreviations, typos, prices, phone numbers and all.

| Column | Meaning |
| --- | --- |
| `label` | `spam` or `ham` |
| `message` | the text of the message |

**Content note.** These are real messages from strangers. Messages with profanity, sexual content, or drug references were removed before this homework, but the remaining spam is still spam, and some ordinary messages are personal.

In [ ]:
# RUN-TOGETHER  (setup)
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

sms = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/sms_spam.csv')
print(sms.shape)
print(sms['label'].value_counts())
print()
print('median number of words per message:')
print(sms['message'].str.split().str.len().groupby(sms['label']).median())
print()
for i in [0, 2]:
    print(sms['label'][i], ':', sms['message'][i])

**Problem 0.** (2 points)

(a) What share of the messages are spam? A filter that never flags anything (Homework 6's v0) would score what accuracy?

(b) How do typical spam and ham messages differ in length? Name two other differences you can see in the two printed messages.

**Your answer:**

(a)

(b)

---
## Part A. From raw text to numbers (10 points)
---

### Problem A1. What should cleaning keep? (3 points)

In [ ]:
# ADAPT  (Problem A1)
# Module 12, Section 2.1: the RUN-TOGETHER cell that lowercases, removes punctuation, and splits a sentence.
# Instead of the made-up sentence, use raw = sms['message'][2] (the spam message printed above).
# Print the cleaned tokens and the token count, as that cell does.

In [ ]:
# RUN-TOGETHER  (Problem A1): how often does a message contain at least one digit?
has_digit = sms['message'].str.contains(r'[0-9]')
print(has_digit.groupby(sms['label']).mean().round(3))

(a) The Module 12 cleaning keeps letters only. List three pieces of the original message that disappeared or were mangled.

(b) Use the second cell: what share of spam messages contain a digit, and what share of ham messages? Was throwing digits away a good idea for *this* task?

(c) `TfidfVectorizer` uses its own tokenizer, which keeps digits (Part B will show the token `150p`, a price of 150 pence). Name one way that keeping digits might also cause trouble.

**Your answer:**

(a)

(b)

(c)

### Problem A2. Stop words: has this data been through it? (3 points)

In [ ]:
# ADAPT  (Problem A2)
# Module 12, Section 2.2: the RUN-TOGETHER cell under "Has our data already been through this?"
# Check these words instead: ['the', 'and', 'to', 'you', 'me', 'my', 'not']
# Use sms['message'].str.lower() in place of reviews['Review'] (these messages are not lowercased).

In [ ]:
# RUN-TOGETHER  (Problem A2): a cleaning step someone else already did
print('messages containing "&lt;#&gt;"     :', sms['message'].str.contains('&lt;#&gt;', regex=False).sum())
print('messages containing "&lt;DECIMAL&gt;":', sms['message'].str.contains('&lt;DECIMAL&gt;', regex=False).sum())
print()
print(sms['message'][sms['message'].str.contains('refilled successfully')].iloc[0])

(a) Compare your output with the TripAdvisor reviews in Module 12, Section 2.2, where `was` and `and` appeared in under 2% of reviews. Have stop words been removed from these messages?

(b) The second cell shows a different kind of earlier cleaning. `&lt;` and `&gt;` are web code for the symbols `<` and `>`. What do you think the people who built this dataset replaced with `<#>` and `<DECIMAL>`, and why might they have done it? (The documentation does not say, so give your best guess.)

**Your answer:**

(a)

(b)

### Problem A3. TF-IDF by hand (4 points)

Three tiny messages, $N = 3$:

$$m_1 = \text{"call me now"} \qquad m_2 = \text{"call now"} \qquad m_3 = \text{"win prize now"}$$

Use the formulas from Module 12, Section 3.2: $\text{TF}(t,d) = \dfrac{\text{count of } t \text{ in } d}{\text{tokens in } d}$, $\text{IDF}(t) = \ln\left(\dfrac{N}{\text{df}(t)}\right)$, and TF-IDF = TF $\times$ IDF.

(a) Fill in df and IDF for every token: `call`, `me`, `now`, `prize`, `win`.

(b) Compute the TF-IDF of every token in $m_3$.

(c) Which token has TF-IDF 0 in every message, and why? Which tokens make $m_3$ look most like spam to a model that weights by TF-IDF?

Check your IDF values with the cell below. Remember the Module 12 note: scikit-learn adds 1 to every IDF and uses raw counts, so its table will not match yours exactly.

**Your answer:**

(a)

| token | df | IDF |
| --- | --- | --- |
| call |  |  |
| me |  |  |
| now |  |  |
| prize |  |  |
| win |  |  |

(b)

(c)

In [ ]:
# FILL-IN  (Problem A3)
tiny = ['call me now', 'call now', 'win prize now']

vec = TfidfVectorizer(norm=None, smooth_idf=False)
M = vec.fit_transform(____)

print(pd.DataFrame(M.toarray(), columns=vec.get_feature_names_out(), index=['m1', 'm2', 'm3']).round(3))
print()
print('scikit-learn IDF (your IDF + 1):')
print(pd.Series(vec.idf_, index=vec.get_feature_names_out()).round(3).to_string())

---
## Part B. Version 5: a text classifier (14 points)
---

### Problem B1. Train it, and decide about stop words (5 points)

In [ ]:
# ADAPT  (Problem B1)
# Module 12, Section 4: the RUN-TOGETHER cell that splits the data, fits `tfidf`, and fits `clf`.
# Changes:
#   - the text is sms['message'] and the label is sms['label']
#   - keep test_size=0.25, random_state=210, stratify on the label, and max_features=2000
#   - keep the names X_train, X_test, y_train, y_test, tfidf, Xtr, Xte, and clf
# Also print the baseline accuracy on the test set: (y_test == 'ham').mean()

Module 12 decided not to remove stop words because the TripAdvisor data had already lost most of them. These messages still have theirs (A2). Test the decision here.

In [ ]:
# FILL-IN  (Problem B1)
tfidf_sw = TfidfVectorizer(max_features=2000, stop_words=____)
Xtr_sw = tfidf_sw.fit_transform(X_train)
Xte_sw = tfidf_sw.transform(X_test)
clf_sw = LogisticRegression(max_iter=2000).fit(Xtr_sw, y_train)

for name, model, Xt in [('keep stop words  ', clf, Xte), ('remove stop words', clf_sw, Xte_sw)]:
    pred = model.predict(Xt)
    caught = (pred[y_test == 'spam'] == 'spam').mean()        # share of the test spam that was flagged
    print(name, '  accuracy:', round(model.score(Xt, y_test), 3), '  spam caught:', round(caught, 3))

(a) Report the test accuracy and the baseline. How much did reading the words buy?

(b) Does removing stop words help or hurt here? Give both numbers for accuracy and for spam caught.

(c) The two words that most strongly mark a message as *ham* in Problem B2 are both stop words. Guess what they are, then check in B2. Why would such words matter more for text messages than for hotel reviews?

**Your answer:**

(a)

(b)

(c)

### Problem B2. Read the model (3 points)

The two classes in alphabetical order are `ham` and `spam`, so a **positive** coefficient pushes a message toward **spam** and a negative one toward ham.

In [ ]:
# ADAPT  (Problem B2)
# Module 12, Section 4.1: the RUN-TOGETHER cell that prints the 10 strongest words in each direction.
# Use clf and tfidf from B1. Change the labels: positive coefficients push toward SPAM, negative toward HAM.

(a) Pick three spam words and say in one phrase what each suggests about how spam is written.

(b) `lt` and `gt` are among the strongest ham words. Using A2, explain where they come from. Module 12 called `told` a fragile feature; are `lt` and `gt` fragile too? What would happen if this filter were used on messages from a phone company that never inserted those placeholders?

**Your answer:**

(a)

(b)

### Problem B3. What kind of mistakes? (3 points)

In [ ]:
# ADAPT  (Problem B3)
# Module 12, Section 4: the RUN-TOGETHER cell with classification_report and the confusion-matrix heatmap.
# Use clf and Xte from B1, and change the title.

(a) How many real messages (ham) did the filter flag as spam? How many spam messages got through? Report the spam precision and spam recall.

(b) In Homework 6, the best filter (the random forest on six numbers) had a spam precision of about 0.89 and a spam recall of about 0.81: roughly 1 flagged email in 9 was a real email. Compare. Which number improved a lot, and which barely changed?

(c) Homework 6 asked whether a filter was safe to **delete** suspected spam automatically. Would you trust version 5 to do that? Use your numbers.

**Your answer:**

(a)

(b)

(c)

### Problem B4. Do bigrams help? (3 points)

In [ ]:
# ADAPT  (Problem B4)
# Module 12, Section 5: the cell that fits tfidf_bi and clf_bi and compares them with the unigram model,
# and then the cell that prints the bigrams pushing hardest in each direction.
# Use X_train, X_test, y_train, y_test from B1. Remember: positive pushes toward SPAM.

(a) Did adding bigrams change the test accuracy?

(b) Pick one spam bigram and one ham bigram and explain why the pair says more than its two words separately. (Hint: compare `you have` with `call me`.)

**Your answer:**

(a)

(b)

---
## Part C. Search and clusters (12 points)
---

### Problem C1. Length, by hand (3 points)

Three messages counted over the vocabulary `free`, `call`, `now`, `me`:

| message | text | free | call | now | me |
| --- | --- | --- | --- | --- | --- |
| $u$ | "free call now" | 1 | 1 | 1 | 0 |
| $v$ | "free call now free call now" | 2 | 2 | 2 | 0 |
| $w$ | "call me now" | 0 | 1 | 1 | 1 |

(a) Compute the Euclidean distances $d(u, v)$ and $d(u, w)$. Which message is "closer" to $u$?

(b) Rescale each message to length 1 (Module 12, Section 6.2) and recompute both distances. Which is closer now?

(c) In one sentence, which answer matches what a person reading the messages would say, and why?

**Your answer:**

(a)

(b)

(c)

### Problem C2. A message search engine (4 points)

In [ ]:
# ADAPT  (Problem C2)
# Module 12, Class Example 12.2: the cell that fits search_vec and builds `docs`, and the cell that defines search().
# Changes:
#   - fit on sms['message'] instead of reviews['Review']
#   - inside search(), copy sms instead of reviews, and print the label and the message instead of the rating and review
# Then run: search('you have won a prize')   and   search('call me later')

(a) What are the labels of the top three results for each query? Did search find the right kind of message?

(b) The three results for `call me later` are word-for-word identical. Run the next cell. What does it tell you about this dataset, and why might the same message appear many times? (Think about what kind of message it is.)

(c) Write one query of your own that you expect to return only spam, and one that you expect to return only ham. Did they work?

**Your answer:**

(a)

(b)

(c)

In [ ]:
# RUN-TOGETHER  (Problem C2)
print('messages that are exact copies of an earlier message:', sms['message'].duplicated().sum())
print(sms[sms['message'].duplicated()]['label'].value_counts())

In [ ]:
# Problem C2(c): your two queries

### Problem C3. What do people text about? (5 points)

In [ ]:
# ADAPT  (Problem C3)
# Module 12, Class Example 12.3: the RUN-TOGETHER cell that fits topic_vec, runs KMeans, and describes each cluster.
# Changes:
#   - use sms['message'] instead of reviews['Review'], and sms instead of reviews
#   - use k = 5 and seed = 210, and keep max_features=2000 and max_df=0.2
#   - instead of the mean rating, print the share of spam in each cluster:
#         (in_cluster['label'] == 'spam').mean()

(a) Name each of the five clusters in two or three words from its top words. Which cluster collects most of the spam, and what share of it is spam?

(b) That spam cluster's top words include `later` and `sorry`, which are not spam words. Using Problem C2(b), explain how they got there. What does this say about k-means on very short messages?

(c) Two clusters are built around something other than meaning: one around the placeholders from A2 and one around texting slang (`ok`, `lor`, `thanx`). Why can such clusters form even though they say little about what the messages are *about*?

(d) Add `stop_words='english'` to the `TfidfVectorizer` line and rerun with seed 210, then with seed 0. Does the spam cluster survive? What does that say about trusting one run of k-means? **Then restore the original settings and rerun the cell.**

**Your answer:**

(a)

(b)

(c)

(d)

---
## Part D. A spam-writing machine (8 points)
---

### Problem D1. Which model wrote it? (4 points)

In [ ]:
# ADAPT  (Problem D1)
# Copy two cells from Module 12, Section 8:
#   1. the RUN-TOGETHER cell in 8.2 that defines tokenize() and count_next_words() (copy only the two functions)
#   2. the RUN-TOGETHER cell in Class Example 12.4 that defines generate() and prints samples A to D
# Changes in the second cell:
#   - build `spam_model` from sms.loc[sms['label'] == 'spam', 'message']
#     and `ham_model`  from sms.loc[sms['label'] == 'ham', 'message']
#   - label the samples 'spam model' and 'ham model'
# Keep rng = np.random.default_rng(210) at the top of the second cell.

(a) Before checking, decide which model wrote each of A to D, and note the word or words that decided it.

(b) Run the next cell to check. Then compare what follows `call` in each model. Explain the difference in one sentence.

In [ ]:
# RUN-TOGETHER  (Problem D1): check your guesses, then look inside the two models
for letter, j in zip('ABCD', order):
    print(letter, '->', samples[j][0])
print()

for name, model in [('spam', spam_model), ('ham', ham_model)]:
    after_call = pd.Series(model[('call',)])
    top5 = (after_call / after_call.sum()).sort_values(ascending=False).head(5)
    print(f'{name} model, most likely words after "call":', top5.round(3).to_dict())

**Your answer:**

(a)

(b)

### Problem D2. Spam copies itself (4 points)

Module 12, Section 8.4 measured how often generated text copies five words in a row from the training text. The cell below does the same for both models.

In [ ]:
# RUN-TOGETHER  (Problem D2)
def five_word_chunks(texts):
    chunks = set()
    for text in texts:
        w = tokenize(text)
        for i in range(len(w) - 4):
            chunks.add(tuple(w[i : i + 5]))
    return chunks

for label in ['spam', 'ham']:
    texts = sms.loc[sms['label'] == label, 'message']
    training_chunks = five_word_chunks(texts)
    for n in [2, 3, 4]:
        model = count_next_words(texts, n=n)
        copied = total = 0
        for _ in range(200):
            w = generate(model, n=n, max_words=40).split()
            for i in range(len(w) - 4):
                total += 1
                copied += tuple(w[i : i + 5]) in training_chunks
        print(f'{label:4}, n = {n}: {100 * copied / total:5.1f}% of generated 5-word chunks copied verbatim')
    print()

(a) For each model, describe how copying changes as $n$ grows. Which Module 10 and Module 12 idea is this?

(b) At every $n$, the spam model copies more than the ham model. Give two reasons, one of them from Problem C2(b).

**Your answer:**

(a)

(b)

---
## Part E. Report to the phone company (4 points)
---

A phone company wants to use your version 5 filter. Write 4 to 6 sentences for a manager who has never trained a model. Say:

- whether you would let it **delete** suspected spam or move it to a **spam folder**, using the numbers from B3;
- one feature of the data (from Parts A to C) that might make the filter work worse on the company's own customers' messages;
- one change to the training data you would ask for before going live.

**Your answer:**